# 03b — Threshold tuning + held-out validation

Consumes `03a_predictions.csv` (the three reasoning_effort runs on the tuning
sample), tunes the F1-optimal threshold per variant, picks a winner, then runs
*only* the winner on the held-out resample (`held_out_batch.csv`) and
reports final test-set metrics.

**The held-out CSV must be regenerated under the new preprocessing first** —
re-run `model_selection/00_Sample_New_Batch.ipynb` once before running this
notebook.  Otherwise the held-out batch will be missing the new feature columns
(FICO, delinq, etc.).

**Outputs (saved to `data/results/llm/`):**
- `03b_predictions.csv` — winner's per-loan predictions on the held-out batch
- `03b_metrics.csv` — final test-set metrics

This notebook does NOT call the LLM until the winner is decided in cell 5,
so re-tuning with a different objective is free (just re-run cells 2–4).

In [ ]:
# llm_utils.py and llm_pricing.py live one directory up — make them importable.
import sys; sys.path.insert(0, "..")

import os
import pandas as pd
import numpy as np
from dotenv import load_dotenv

from llm_utils import (
    load_llm_sample, run_ml_on_sample, run_llm_experiment,
    find_best_threshold, evaluate_predictions, compare_results,
    DATA_DIR, RESULTS_DIR,
)

load_dotenv("../.env", override=False)

In [ ]:
# Load 05's tuning-set predictions/probs for all three variants.
tuning = pd.read_csv(f"{RESULTS_DIR}/03a_predictions.csv")
print(f"Loaded {len(tuning)} rows from 03a_predictions.csv")
print(f"  variants: {sorted(tuning['reasoning_effort'].unique())}")
print(f"  loans per variant: {tuning.groupby('reasoning_effort').size().to_dict()}")

In [ ]:
# Per-variant threshold tuning analysis.
# For each variant: F1 at the LLM's default hard 0/1 (= threshold 0.5 on probs)
# vs F1 at the val-tuned threshold maximising minority-class F1.

rows = []
for effort, sub in tuning.groupby('reasoning_effort'):
    y_true = sub['actual'].values
    probs  = sub['prob_fully_paid'].values
    preds_default = sub['llm_pred'].values

    # Drop rows where prob is missing (parse error or no logprobs).
    mask = ~pd.isna(probs)
    y_true_v = y_true[mask]
    probs_v  = probs[mask]
    preds_v  = preds_default[mask]

    from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                                  recall_score, roc_auc_score)

    # Default-threshold metrics (whatever the LLM hard-output)
    acc_def = accuracy_score(y_true_v, preds_v)
    f1_def  = f1_score(y_true_v, preds_v, pos_label=0, zero_division=0)
    auc     = roc_auc_score(y_true_v, probs_v) if len(set(y_true_v)) == 2 else float('nan')

    # Tuned threshold (optimises F1 on Charged Off)
    t, p_co, r_co, f1_tuned = find_best_threshold(y_true_v, probs_v)
    preds_tuned = (probs_v >= t).astype(int)
    acc_tuned = accuracy_score(y_true_v, preds_tuned)

    rows.append({
        'reasoning_effort': effort,
        'n':                len(y_true_v),
        'auc':              auc,
        'f1_default':       f1_def,
        'f1_tuned':         f1_tuned,
        'threshold_tuned':  t,
        'acc_default':      acc_def,
        'acc_tuned':        acc_tuned,
        'precision_co_tuned': p_co,
        'recall_co_tuned':    r_co,
    })

tuning_table = pd.DataFrame(rows).set_index('reasoning_effort').sort_index()
print(tuning_table.to_string(float_format=lambda x: f'{x:.4f}'))

## Decision rule

Pick the winner manually after looking at the table above.  Suggested heuristic:

- If `f1_tuned` and `auc` for `high` exceed those of `medium` by **≥ 0.02 F1 AND ≥ 0.005 AUC** → **high** wins.
- Otherwise → **medium** wins (cheaper per call: ~5–10× less than high).
- `low` is included for completeness; rarely the right pick unless cost is the dominant constraint.

Set `WINNER_REASONING_EFFORT` below to the chosen variant.  This is the only
cell you should edit between the table above and the held-out test below.

In [ ]:
# === EDIT THIS ONE LINE AFTER LOOKING AT THE TABLE ABOVE ===
WINNER_REASONING_EFFORT = "medium"   # one of: "low", "medium", "high"
# ===========================================================

assert WINNER_REASONING_EFFORT in {"low", "medium", "high"}
WINNER_THRESHOLD = float(tuning_table.loc[WINNER_REASONING_EFFORT, 'threshold_tuned'])
print(f"Winner: GPT-5 reasoning={WINNER_REASONING_EFFORT}")
print(f"Tuned threshold (from tuning sample): {WINNER_THRESHOLD:.4f}")

In [ ]:
# Load the held-out resample. This file is INPUT data (the test set);
# it lives in data/processed/ and is generated by 00_Sample_New_Batch.ipynb.
held_out_path = f"{DATA_DIR}/held_out_batch.csv"
assert os.path.exists(held_out_path), (
    f"Held-out batch not found at {held_out_path}. "
    "Re-run model_selection/00_Sample_New_Batch.ipynb first to regenerate it "
    "under the new preprocessing (with FICO, delinq, etc.)."
)
held_out = pd.read_csv(held_out_path)
y_true_held = held_out['loan_status'].values
print(f"Held-out batch: {len(held_out)} loans")
print(f"  Charged Off: {(y_true_held == 0).sum()}")
print(f"  Fully Paid:  {(y_true_held == 1).sum()}")

# XGBoost on the held-out batch (for context comparison)
xgb_probs_held, xgb_preds_held = run_ml_on_sample(held_out)
xgb_metrics_held = evaluate_predictions(
    y_true_held, xgb_preds_held.tolist(),
    label="XGBoost (held-out)", probabilities=xgb_probs_held.tolist(),
)

In [ ]:
# Run the winner on the held-out batch (single API key — no parallelism needed
# for one variant). This is the only API spend in 06.
WINNER_API_KEY = os.environ.get("OPENAI_API_KEY")  # any of the three works

winner_result = run_llm_experiment(
    held_out,
    api_provider="openai",
    model_name="gpt-5",
    api_key=WINNER_API_KEY,
    label=f"GPT-5 reasoning={WINNER_REASONING_EFFORT} (held-out)",
    include_desc=False,
    with_logprobs=True,
    reasoning_effort=WINNER_REASONING_EFFORT,
)

In [ ]:
# Apply the winner's tuning-set threshold to the held-out probabilities.
# This is the headline number: out-of-sample F1 at a fairly-tuned threshold.
held_probs = np.array([p if p is not None else np.nan for p in winner_result['probabilities']])
mask = ~np.isnan(held_probs)
held_y_true_v = y_true_held[mask]
held_probs_v  = held_probs[mask]

held_preds_default = np.array(winner_result['predictions'])[mask]
held_preds_tuned   = (held_probs_v >= WINNER_THRESHOLD).astype(int)

from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                              recall_score, roc_auc_score, classification_report,
                              confusion_matrix)

print(f"\n{'='*60}")
print(f"Held-out test results — GPT-5 reasoning={WINNER_REASONING_EFFORT}")
print(f"Threshold from tuning sample: {WINNER_THRESHOLD:.4f}")
print(f"{'='*60}\n")

print("At LLM default (hard 0/1):")
print(f"  Accuracy:  {accuracy_score(held_y_true_v, held_preds_default)*100:.1f}%")
print(f"  AUC:       {roc_auc_score(held_y_true_v, held_probs_v):.4f}")
print(f"  F1 (CO):   {f1_score(held_y_true_v, held_preds_default, pos_label=0, zero_division=0):.4f}")
print(f"  Recall:    {recall_score(held_y_true_v, held_preds_default, pos_label=0, zero_division=0):.4f}")
print(f"  Precision: {precision_score(held_y_true_v, held_preds_default, pos_label=0, zero_division=0):.4f}")

print(f"\nAt val-tuned threshold ({WINNER_THRESHOLD:.4f}):")
print(f"  Accuracy:  {accuracy_score(held_y_true_v, held_preds_tuned)*100:.1f}%")
print(f"  AUC:       {roc_auc_score(held_y_true_v, held_probs_v):.4f}  (threshold-independent)")
print(f"  F1 (CO):   {f1_score(held_y_true_v, held_preds_tuned, pos_label=0, zero_division=0):.4f}")
print(f"  Recall:    {recall_score(held_y_true_v, held_preds_tuned, pos_label=0, zero_division=0):.4f}")
print(f"  Precision: {precision_score(held_y_true_v, held_preds_tuned, pos_label=0, zero_division=0):.4f}")

print(f"\nClassification report (tuned threshold):")
print(classification_report(held_y_true_v, held_preds_tuned,
                            target_names=['Charged Off', 'Fully Paid']))
print("Confusion matrix (tuned threshold):")
print(confusion_matrix(held_y_true_v, held_preds_tuned))

In [ ]:
# Save consolidated outputs.
# 03b_predictions.csv: per-loan winner predictions on held-out, both threshold strategies.
predictions_out = pd.DataFrame({
    'row_index':         range(len(held_out)),
    'reasoning_effort':  WINNER_REASONING_EFFORT,
    'actual':            y_true_held,
    'llm_pred_default':  winner_result['predictions'],
    'llm_pred_tuned':    [int(p >= WINNER_THRESHOLD) if p is not None and not pd.isna(p) else None
                          for p in winner_result['probabilities']],
    'prob_fully_paid':   winner_result['probabilities'],
    'llm_reasoning':     winner_result['reasonings'],
    'xgb_pred':          xgb_preds_held.tolist(),
    'xgb_prob':          xgb_probs_held.tolist(),
})
predictions_out.to_csv(f"{RESULTS_DIR}/03b_predictions.csv", index=False)

# 03b_metrics.csv: side-by-side LLM vs XGBoost on held-out.
metrics_out = pd.DataFrame([
    {
        'variant': f"GPT-5 reasoning={WINNER_REASONING_EFFORT} (default)",
        'accuracy': accuracy_score(held_y_true_v, held_preds_default),
        'auc':      roc_auc_score(held_y_true_v, held_probs_v),
        'f1_co':    f1_score(held_y_true_v, held_preds_default, pos_label=0, zero_division=0),
        'recall_co':    recall_score(held_y_true_v, held_preds_default, pos_label=0, zero_division=0),
        'precision_co': precision_score(held_y_true_v, held_preds_default, pos_label=0, zero_division=0),
        'threshold': 0.5,
    },
    {
        'variant': f"GPT-5 reasoning={WINNER_REASONING_EFFORT} (tuned)",
        'accuracy': accuracy_score(held_y_true_v, held_preds_tuned),
        'auc':      roc_auc_score(held_y_true_v, held_probs_v),
        'f1_co':    f1_score(held_y_true_v, held_preds_tuned, pos_label=0, zero_division=0),
        'recall_co':    recall_score(held_y_true_v, held_preds_tuned, pos_label=0, zero_division=0),
        'precision_co': precision_score(held_y_true_v, held_preds_tuned, pos_label=0, zero_division=0),
        'threshold': WINNER_THRESHOLD,
    },
    {
        'variant': "XGBoost (held-out)",
        **{k: xgb_metrics_held.get(k) for k in ['accuracy','auc','precision_charged_off',
                                                 'recall_charged_off','f1_charged_off']},
    },
])
metrics_out.to_csv(f"{RESULTS_DIR}/03b_metrics.csv", index=False)

print(f"Saved 03b_predictions.csv ({len(predictions_out)} rows)")
print(f"Saved 03b_metrics.csv ({len(metrics_out)} rows)")